In [24]:
from pyspark.sql import DataFrame, functions as F, Window
from delta.tables import DeltaTable

SOURCE_TABLE = "silver.sales_obt"
GOLD_SCHEMA = "gold"

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 26, Finished, Available, Finished, False)

In [25]:
DIMENSIONS = [
    {
        "dimension_name": "Customers",
        "target_table": "dim_customers",
        "business_key": "customer_id",
        "surrogate_key": "customer_sk",
        "source_columns": [
            "customer_id",
            "customer_first_name", "customer_last_name", "customer_email",
            "customer_phone", "customer_city", "customer_province", "customer_country",
            "customer_created_timestamp", "customer_updated_timestamp",
            "customer_is_active",
        ],
        "gold_processed_at_column": "customer_processed_at",
        "tracked_attributes": [
            "customer_first_name", "customer_last_name", "customer_email",
            "customer_phone", "customer_city", "customer_province", "customer_country",
        ],
    },
    {
        "dimension_name": "Employees",
        "target_table": "dim_employees",
        "business_key": "employee_id",
        "surrogate_key": "employee_sk",
        "source_columns": [
            "employee_id",
            "employee_first_name", "employee_last_name", "employee_email",
            "job_title", "salary", "store_id",
            "employee_created_timestamp", "employee_updated_timestamp",
            "employee_is_active",
        ],
        "gold_processed_at_column": "employee_processed_at",
        "tracked_attributes": [
            "employee_first_name", "employee_last_name", "employee_email",
            "job_title", "salary", "store_id",
        ],
    },
    {
        "dimension_name": "Orders",
        "target_table": "dim_orders",
        "business_key": "order_id",
        "surrogate_key": "order_sk",
        "source_columns": [
            "order_id", "order_item_id",
            "payment_method", "order_status", "order_timestamp",
            "order_created_timestamp", "order_updated_timestamp",
            "order_is_active", "obt_processed_at",
        ],
        "gold_processed_at_column": "order_processed_at",
        "tracked_attributes": ["payment_method", "order_status", "order_timestamp"],
    },
    {
        "dimension_name": "Products",
        "target_table": "dim_products",
        "business_key": "product_id",
        "surrogate_key": "product_sk",
        "source_columns": [
            "product_id",
            "product_name", "category", "brand", "price",
            "product_created_timestamp", "product_updated_timestamp",
            "product_is_active",
        ],
        "gold_processed_at_column": "product_processed_at",
        "tracked_attributes": ["product_name", "category", "brand", "price"],
    },
    {
        "dimension_name": "Stores",
        "target_table": "dim_stores",
        "business_key": "store_id",
        "surrogate_key": "store_sk",
        "source_columns": [
            "store_id",
            "store_name", "store_city", "store_province", "store_country",
            "store_created_timestamp", "store_updated_timestamp",
            "store_is_active",
        ],
        "gold_processed_at_column": "store_processed_at",
        "tracked_attributes": ["store_name", "store_city", "store_province", "store_country"],
    },
]

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 27, Finished, Available, Finished, False)

In [26]:
def create_dimension_source(dim_meta: dict, source_table: str = SOURCE_TABLE) -> DataFrame:
    columns_sql = ",\n    ".join(dim_meta["source_columns"])
    gold_col = dim_meta["gold_processed_at_column"]
 
    sql = (
        f"SELECT DISTINCT\n"
        f"    {columns_sql},\n"
        f"    CURRENT_TIMESTAMP() AS {gold_col}\n"
        f"FROM {source_table}"
    )
 
    print(f"[SOURCE SQL] {dim_meta['dimension_name']}")
    print(sql)
 
    return spark.sql(sql)
 
 
def add_hash_column(df: DataFrame, tracked_attributes: list, hash_column: str = "row_hash") -> DataFrame:
    concat_expr = F.concat_ws(
        "||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("§NULL§")) for c in tracked_attributes]
    )
    return df.withColumn(hash_column, F.sha2(concat_expr, 256))
 
 
def get_max_surrogate_key(target_table: str, surrogate_key_col: str) -> int:
    if not spark.catalog.tableExists(target_table):
        return 0
    max_val = spark.table(target_table).agg(F.max(F.col(surrogate_key_col))).collect()[0][0]
    return int(max_val) if max_val is not None else 0
 
 
def _dimension_column_order(dim_meta: dict) -> list:
    return (
        [dim_meta["surrogate_key"]]
        + dim_meta["source_columns"]
        + [dim_meta["gold_processed_at_column"], "row_hash",
           "valid_from", "valid_to", "is_current",
           "created_at", "updated_at"]
    )
 
 
def create_scd2_merge(source_df: DataFrame, dim_meta: dict, schema: str = GOLD_SCHEMA) -> dict:
    target_table = f"{schema}.{dim_meta['target_table']}"
    business_key = dim_meta["business_key"]
    hash_col = "row_hash"
 
    source_hashed = add_hash_column(source_df, dim_meta["tracked_attributes"], hash_col)
 
    if not spark.catalog.tableExists(target_table):
        print(f"[CREATE] {target_table} does not exist — performing initial load")
        window = Window.orderBy(business_key)
        initial_df = (
            source_hashed
            .withColumn(dim_meta["surrogate_key"], F.row_number().over(window))
            .withColumn("valid_from", F.current_timestamp())
            .withColumn("valid_to", F.lit(None).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
            .select(_dimension_column_order(dim_meta))
        )
        (
            initial_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
        )
        row_count = initial_df.count()
        return {"inserted": row_count, "updated": 0, "expired": 0}
 
    current_target = (
        spark.table(target_table)
             .filter(F.col("is_current") == True)  # noqa: E712
             .select(business_key, F.col(hash_col).alias("target_hash"))
    )
 

    joined = source_hashed.join(current_target, on=business_key, how="left").cache()
 
    changed_count = joined.filter(
        F.col("target_hash").isNotNull() & (F.col(hash_col) != F.col("target_hash"))
    ).count()
    new_count = joined.filter(F.col("target_hash").isNull()).count()
 
    to_insert_df = joined.filter(
        F.col("target_hash").isNull() | (F.col(hash_col) != F.col("target_hash"))
    ).drop("target_hash")
 
    delta_target = DeltaTable.forName(spark, target_table)
    (
        delta_target.alias("t")
        .merge(
            source_hashed.alias("s"),
            f"t.{business_key} = s.{business_key} AND t.is_current = true"
        )
        .whenMatchedUpdate(
            condition=f"t.{hash_col} <> s.{hash_col}",
            set={
                "is_current": F.lit(False),
                "valid_to": F.current_timestamp(),
                "updated_at": F.current_timestamp(),
            },
        )
        .execute()
    )
 
    if changed_count + new_count > 0:
        max_sk = get_max_surrogate_key(target_table, dim_meta["surrogate_key"])
        window = Window.orderBy(business_key)
        to_insert_df = (
            to_insert_df
            .withColumn(dim_meta["surrogate_key"], F.row_number().over(window) + F.lit(max_sk))
            .withColumn("valid_from", F.current_timestamp())
            .withColumn("valid_to", F.lit(None).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
            .select(_dimension_column_order(dim_meta))
        )
        (
            to_insert_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table)
        )
 
    joined.unpersist()
 
    return {
        "inserted": changed_count + new_count,
        "updated": changed_count,
        "expired": changed_count,
    }
 
 
def process_dimension(dim_meta: dict, source_table: str = SOURCE_TABLE, schema: str = GOLD_SCHEMA) -> dict:
    print("=" * 80)
    print(f"Processing dimension: {dim_meta['dimension_name']}")
    print("=" * 80)
 
    source_df = create_dimension_source(dim_meta, source_table)
    result = create_scd2_merge(source_df, dim_meta, schema)
 
    print(
        f"[RESULT] {schema}.{dim_meta['target_table']}: "
        f"inserted={result['inserted']}, updated={result['updated']}, expired={result['expired']}"
    )
 
    return {"dimension": dim_meta["target_table"], **result}

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 28, Finished, Available, Finished, False)

In [27]:
processing_results = [process_dimension(dim_meta) for dim_meta in DIMENSIONS]

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 29, Finished, Available, Finished, False)

Processing dimension: Customers
[SOURCE SQL] Customers
SELECT DISTINCT
    customer_id,
    customer_first_name,
    customer_last_name,
    customer_email,
    customer_phone,
    customer_city,
    customer_province,
    customer_country,
    customer_created_timestamp,
    customer_updated_timestamp,
    customer_is_active,
    CURRENT_TIMESTAMP() AS customer_processed_at
FROM silver.sales_obt
[CREATE] gold.dim_customers does not exist — performing initial load
[RESULT] gold.dim_customers: inserted=1991, updated=0, expired=0
Processing dimension: Employees
[SOURCE SQL] Employees
SELECT DISTINCT
    employee_id,
    employee_first_name,
    employee_last_name,
    employee_email,
    job_title,
    salary,
    store_id,
    employee_created_timestamp,
    employee_updated_timestamp,
    employee_is_active,
    CURRENT_TIMESTAMP() AS employee_processed_at
FROM silver.sales_obt
[CREATE] gold.dim_employees does not exist — performing initial load
[RESULT] gold.dim_employees: inserted=25

In [28]:
processing_results = [process_dimension(dim_meta) for dim_meta in DIMENSIONS]

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 30, Finished, Available, Finished, False)

Processing dimension: Customers
[SOURCE SQL] Customers
SELECT DISTINCT
    customer_id,
    customer_first_name,
    customer_last_name,
    customer_email,
    customer_phone,
    customer_city,
    customer_province,
    customer_country,
    customer_created_timestamp,
    customer_updated_timestamp,
    customer_is_active,
    CURRENT_TIMESTAMP() AS customer_processed_at
FROM silver.sales_obt
[RESULT] gold.dim_customers: inserted=0, updated=0, expired=0
Processing dimension: Employees
[SOURCE SQL] Employees
SELECT DISTINCT
    employee_id,
    employee_first_name,
    employee_last_name,
    employee_email,
    job_title,
    salary,
    store_id,
    employee_created_timestamp,
    employee_updated_timestamp,
    employee_is_active,
    CURRENT_TIMESTAMP() AS employee_processed_at
FROM silver.sales_obt
[RESULT] gold.dim_employees: inserted=0, updated=0, expired=0
Processing dimension: Orders
[SOURCE SQL] Orders
SELECT DISTINCT
    order_id,
    order_item_id,
    payment_method,
 

In [29]:
summary_df = spark.createDataFrame(processing_results)
print("=" * 80)
print("GOLD DIMENSION PROCESSING SUMMARY")
print("=" * 80)
display(summary_df)

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 31, Finished, Available, Finished, False)

GOLD DIMENSION PROCESSING SUMMARY


SynapseWidget(Synapse.DataFrame, b40011e7-696e-4936-9f16-46c230d7d9d1)

In [30]:
%%sql
select * from silver.sales_obt limit 2

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 32, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 58 fields>

In [33]:
%%sql
select * from gold.dim_customers limit 4;

StatementMeta(, d339f097-73b3-4acc-abe2-749631bae44c, 35, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 19 fields>